In [ ]:
# ============================================================
# 02b_rich_tensor.ipynb
# Build enriched hourly tensor X_rich.npy
# Shape: (n_stays, 24, 17)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import psutil
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────
DATA_DIR   = Path('/mnt/c/Users/20220460/Desktop/gp/Cleaned')
OUTPUT_DIR = Path('/mnt/c/Users/20220460/Desktop/mimic_iv_processed')

FEATURE_COLS = [
    'abp_dia', 'abp_mean', 'abp_sys',
    'heart_rate', 'resp_rate', 'spo2', 'temp_c'
]

RICH_FEATURE_COLS = [
    'abp_dia',          # 0
    'abp_mean',         # 1
    'abp_sys',          # 2
    'heart_rate',       # 3
    'resp_rate',        # 4
    'spo2',             # 5
    'temp_c',           # 6
    'lactate',          # 7
    'wbc',              # 8
    'sofa_platelets',   # 9
    'sofa_bilirubin',   # 10
    'sofa_creatinine',  # 11
    'urine_output',     # 12
    'vasopressor_flag', # 13
    'shock_index',      # 14
    'hr_delta',         # 15
    'temp_deviation',   # 16
    'crp',              # 17  ← new
    'platelets_raw',    # 18  ← new (raw value, not SOFA score)
    'inr',              # 19  ← new
    'lactate_fresh',    # 20  ← new freshness flag
    'wbc_fresh',        # 21  ← new freshness flag
    'crp_fresh',        # 22  ← new freshness flag
    'platelets_fresh',  # 23  ← new freshness flag
    'inr_fresh',        # 24  ← new freshness flag
]

print("=" * 55)
print("02b — Rich Hourly Tensor Builder")
print("=" * 55)

# ── Load base data ────────────────────────────────────────────
print("\nLoading base data...")

cohort = pd.read_csv(OUTPUT_DIR / 'icu_cohort.csv')
cohort['intime']  = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])
print(f"  Cohort        : {cohort.shape} | "
      f"{cohort['stay_id'].nunique():,} stays")

X_vitals = np.load(OUTPUT_DIR / 'X_vitals.npy')
print(f"  X_vitals      : {X_vitals.shape}")

stay_ids_order = pd.read_csv(OUTPUT_DIR / 'hourly_labels.csv')\
    .sort_values(['stay_id', 'hour'])['stay_id']\
    .drop_duplicates()\
    .tolist()
stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}
print(f"  Stay order    : {len(stay_ids_order):,} stays")

# Memory check
mem = psutil.virtual_memory()
print(f"\nMemory available: {mem.available / 1e9:.1f} GB "
      f"/ {mem.total / 1e9:.1f} GB total")
print("\nBase data loaded ✓")

02b — Rich Hourly Tensor Builder

Loading base data...
  Cohort        : (93730, 28) | 93,730 stays
  X_vitals      : (89075, 24, 7)
  Stay order    : 74,550 stays

Memory available: 4.1 GB / 16.7 GB total

Base data loaded ✓


## Load and process extra labs (lactate+WBC)

In [3]:
# ── Cell 2: Load and process extra labs (lactate + WBC) ───────
print("Loading extra labs (lactate + WBC)...")

extra_labs = pd.read_csv(OUTPUT_DIR / 'extra_labs_filtered.csv')
extra_labs['charttime'] = pd.to_datetime(extra_labs['charttime'])
print(f"  Raw shape     : {extra_labs.shape}")
print(f"  Columns       : {extra_labs.columns.tolist()}")
print(f"  Item IDs      : {extra_labs['itemid'].value_counts().to_dict()}")
print(f"  Missing       : {extra_labs.isnull().sum().to_dict()}")
extra_labs.head(3)

Loading extra labs (lactate + WBC)...
  Raw shape     : (1822267, 6)
  Columns       : ['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum', 'stay_id']
  Item IDs      : {51301: 1329151, 50813: 493116}
  Missing       : {'subject_id': 0, 'hadm_id': 0, 'itemid': 0, 'charttime': 0, 'valuenum': 4214, 'stay_id': 0}


,subject_id,hadm_id,itemid,charttime,valuenum,stay_id
0,10000032,29079034.0,51301,2180-07-24 06:35:00,4.1,39553978
1,10000032,29079034.0,51301,2180-07-25 04:45:00,4.8,39553978
2,10000690,25860671.0,51301,2150-11-03 02:56:00,7.5,37081114


In [ ]:
# ── Cell 3: Process extra labs into hourly lookup ─────────────
print("Processing extra labs into hourly format...")

ITEMID_TO_NAME = {
    50813: 'lactate',
    52442: 'lactate',
    51301: 'wbc',
    50889: 'crp',
    51265: 'platelets_raw',
    51237: 'inr',
}

# Map names — keep NaNs, do NOT drop them
extra_labs['lab_name'] = extra_labs['itemid'].map(ITEMID_TO_NAME)

# Drop rows where valuenum is NaN only if the measurement
# was never recorded (not just missing at that hour)
# We drop NaN valuenum because there's no value to store
extra_labs = extra_labs.dropna(subset=['valuenum'])

# Merge intime
extra_labs = extra_labs.merge(
    cohort[['stay_id', 'intime']], on='stay_id', how='left'
)
extra_labs['intime'] = pd.to_datetime(extra_labs['intime'])

# Compute hour since admission
extra_labs['hour'] = (
    (extra_labs['charttime'] - extra_labs['intime'])
    .dt.total_seconds() / 3600
).astype(int)

# Keep first 24h only
extra_labs = extra_labs[
    (extra_labs['hour'] >= 0) &
    (extra_labs['hour'] < 24)
]

# Aggregate multiple readings in same hour → mean
# (e.g. if lactate was drawn twice in hour 3, take mean)
extra_labs_hourly = (
    extra_labs
    .groupby(['stay_id', 'hour', 'lab_name'])['valuenum']
    .mean()
    .reset_index()
)

# Pivot wide — hours with no reading will be NaN naturally
extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'hour'],
    columns='lab_name',
    values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None

# Ensure both columns exist even if one lab was never recorded
for col in ['lactate', 'wbc', 'crp', 'platelets_raw', 'inr']:
    if col not in extra_labs_wide.columns:
        extra_labs_wide[col] = np.nan

print(f"  Shape after processing : {extra_labs_wide.shape}")
print(f"  Stays covered          : {extra_labs_wide['stay_id'].nunique():,}")
print(f"  Missing lactate        : {extra_labs_wide['lactate'].isnull().sum():,}")
print(f"  Missing wbc            : {extra_labs_wide['wbc'].isnull().sum():,}")
print(f"  Sample:")
print(extra_labs_wide.head(5).to_string())

# Build stay-level lookup: {stay_id: {hour: {lactate: v, wbc: v}}}
# Hours with no reading simply won't appear in the dict
# → tensor will remain NaN for those hours
# → imputation happens in file 3 dataset class
labs_lookup = {}
for row in extra_labs_wide.itertuples():
    sid = row.stay_id
    if sid not in labs_lookup:
        labs_lookup[sid] = {}
    labs_lookup[sid][int(row.hour)] = {
    'lactate'     : row.lactate      if not pd.isna(row.lactate)      else np.nan,
    'wbc'         : row.wbc          if not pd.isna(row.wbc)          else np.nan,
    'crp'         : row.crp          if not pd.isna(row.crp)          else np.nan,
    'platelets_raw': row.platelets_raw if not pd.isna(row.platelets_raw) else np.nan,
    'inr'         : row.inr          if not pd.isna(row.inr)          else np.nan,
}

print(f"\n  Lookup built for {len(labs_lookup):,} stays")

# Free memory
del extra_labs, extra_labs_hourly, extra_labs_wide
import gc; gc.collect()
print("  Memory freed ✓")
print("Extra labs processed ✓")

Processing extra labs into hourly format...
  Shape after processing : (280659, 4)
  Stays covered          : 91,110
  Missing lactate        : 135,070
  Missing wbc            : 92,313
  Sample:
    stay_id  hour  lactate   wbc
0  30000153     1      1.7   NaN
1  30000153     3      NaN  17.0
2  30000153    15      NaN  15.2
3  30000213     2      0.9   NaN
4  30000213    22      NaN   5.8

  Lookup built for 91,110 stays
  Memory freed ✓
Extra labs processed ✓


## Process SOFA Labs

In [5]:
# ── Cell 4: Process SOFA labs into hourly lookup ──────────────
print("Loading SOFA labs...")

sofa_labs = pd.read_csv(OUTPUT_DIR / 'sofa_labs_hourly_wide.csv')
sofa_labs['charttime_hour'] = pd.to_datetime(sofa_labs['charttime_hour'])
print(f"  Raw shape  : {sofa_labs.shape}")
print(f"  Columns    : {sofa_labs.columns.tolist()}")
print(f"  Sample:")
print(sofa_labs.head(3).to_string())

Loading SOFA labs...
  Raw shape  : (1623260, 5)
  Columns    : ['stay_id', 'charttime_hour', 'bilirubin', 'creatinine', 'platelets']
  Sample:
    stay_id      charttime_hour  bilirubin  creatinine  platelets
0  30000153 2174-09-29 15:00:00        NaN         0.9      173.0
1  30000153 2174-09-30 03:00:00        NaN         1.1      162.0
2  30000153 2174-10-01 05:00:00        NaN         0.8      112.0


In [6]:
# ── Cell 4: Process SOFA labs into hourly lookup ──────────────
print("Processing SOFA labs...")

def score_platelets(x):
    if pd.isna(x): return np.nan
    if x >= 150:   return 0
    if x >= 100:   return 1
    if x >= 50:    return 2
    if x >= 20:    return 3
    return 4

def score_bilirubin(x):
    if pd.isna(x): return np.nan
    if x < 1.2:    return 0
    if x < 2.0:    return 1
    if x < 6.0:    return 2
    if x < 12.0:   return 3
    return 4

def score_creatinine(x):
    if pd.isna(x): return np.nan
    if x < 1.2:    return 0
    if x < 2.0:    return 1
    if x < 3.5:    return 2
    if x < 5.0:    return 3
    return 4

# Merge intime to compute hours since admission
sofa_labs = sofa_labs.merge(
    cohort[['stay_id', 'intime']], on='stay_id', how='left'
)
sofa_labs['intime'] = pd.to_datetime(sofa_labs['intime'])
sofa_labs['hour']   = (
    (sofa_labs['charttime_hour'] - sofa_labs['intime'])
    .dt.total_seconds() / 3600
).astype(int)

# Keep first 24h only
sofa_labs = sofa_labs[
    (sofa_labs['hour'] >= 0) &
    (sofa_labs['hour'] < 24)
].copy()

# Compute SOFA scores — NaN stays NaN if raw value is NaN
sofa_labs['sofa_platelets']  = sofa_labs['platelets'].apply(score_platelets)
sofa_labs['sofa_bilirubin']  = sofa_labs['bilirubin'].apply(score_bilirubin)
sofa_labs['sofa_creatinine'] = sofa_labs['creatinine'].apply(score_creatinine)

# Aggregate to hourly mean per stay
sofa_hourly = (
    sofa_labs
    .groupby(['stay_id', 'hour'])[
        ['sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine']
    ]
    .mean()
    .reset_index()
)

print(f"  Shape after processing : {sofa_hourly.shape}")
print(f"  Stays covered          : {sofa_hourly['stay_id'].nunique():,}")
print(f"  Missing sofa_platelets : {sofa_hourly['sofa_platelets'].isnull().sum():,}")
print(f"  Missing sofa_bilirubin : {sofa_hourly['sofa_bilirubin'].isnull().sum():,}")
print(f"  Missing sofa_creatinine: {sofa_hourly['sofa_creatinine'].isnull().sum():,}")
print(f"  Sample:")
print(sofa_hourly.head(5).to_string())

# Build lookup: {stay_id: {hour: {sofa_platelets: v, ...}}}
sofa_lookup = {}
for row in sofa_hourly.itertuples():
    sid = row.stay_id
    if sid not in sofa_lookup:
        sofa_lookup[sid] = {}
    sofa_lookup[sid][int(row.hour)] = {
        'sofa_platelets'  : row.sofa_platelets,
        'sofa_bilirubin'  : row.sofa_bilirubin,
        'sofa_creatinine' : row.sofa_creatinine,
    }

print(f"\n  Lookup built for {len(sofa_lookup):,} stays")

# Free memory
del sofa_labs, sofa_hourly
import gc; gc.collect()
print("  Memory freed ✓")
print("SOFA labs processed ✓")

Processing SOFA labs...
  Shape after processing : (227466, 5)
  Stays covered          : 91,372
  Missing sofa_platelets : 35,931
  Missing sofa_bilirubin : 167,968
  Missing sofa_creatinine: 36,763
  Sample:
    stay_id  hour  sofa_platelets  sofa_bilirubin  sofa_creatinine
0  30000153     2             0.0             NaN              0.0
1  30000153    14             0.0             NaN              0.0
2  30000213     8             NaN             NaN              3.0
3  30000213    14             NaN             NaN              3.0
4  30000213    22             0.0             0.0              3.0

  Lookup built for 91,372 stays
  Memory freed ✓
SOFA labs processed ✓


## Urine Output

In [7]:
# ── Cell 5: Process urine output into hourly lookup ───────────
print("Loading urine output...")

uo = pd.read_csv(OUTPUT_DIR / 'urine_output_filtered.csv')
print(f"  Raw shape  : {uo.shape}")
print(f"  Columns    : {uo.columns.tolist()}")
print(f"  Sample:")
print(uo.head(3).to_string())

Loading urine output...
  Raw shape  : (4143759, 4)
  Columns    : ['stay_id', 'charttime', 'itemid', 'value']
  Sample:
    stay_id            charttime  itemid  value
0  39553978  2180-07-23 15:00:00  226560  175.0
1  37081114  2150-11-06 08:00:00  226559   15.0
2  37081114  2150-11-06 09:00:00  226559   15.0


In [8]:
# ── Cell 5: Process urine output into hourly lookup ───────────
print("Processing urine output...")

uo['charttime'] = pd.to_datetime(uo['charttime'])

# Merge intime
uo = uo.merge(
    cohort[['stay_id', 'intime']], on='stay_id', how='left'
)
uo['intime'] = pd.to_datetime(uo['intime'])

# Compute hour since admission
uo['hour'] = (
    (uo['charttime'] - uo['intime'])
    .dt.total_seconds() / 3600
).astype(int)

# Keep first 24h only
uo = uo[
    (uo['hour'] >= 0) &
    (uo['hour'] < 24)
].copy()

# Clean value column
uo['value'] = pd.to_numeric(uo['value'], errors='coerce')

# Clip physiologically impossible values
# Negative = data correction entries
# Max realistic hourly UO ~500 mL/hr
uo['value'] = uo['value'].clip(lower=0, upper=500)

# Aggregate to hourly sum per stay
# UO is a volume measurement — sum within hour makes more
# sense than mean (multiple outputs in same hour add up)
uo_hourly = (
    uo
    .groupby(['stay_id', 'hour'])['value']
    .sum()
    .reset_index()
    .rename(columns={'value': 'urine_output'})
)

print(f"  Shape after processing : {uo_hourly.shape}")
print(f"  Stays covered          : {uo_hourly['stay_id'].nunique():,}")
print(f"  Missing urine_output   : {uo_hourly['urine_output'].isnull().sum():,}")
print(f"  UO stats (mL/hr):")
print(f"    Mean  : {uo_hourly['urine_output'].mean():.1f}")
print(f"    Median: {uo_hourly['urine_output'].median():.1f}")
print(f"    Max   : {uo_hourly['urine_output'].max():.1f}")
print(f"  Sample:")
print(uo_hourly.head(5).to_string())

# Build lookup: {stay_id: {hour: urine_output}}
uo_lookup = {}
for row in uo_hourly.itertuples():
    sid = row.stay_id
    if sid not in uo_lookup:
        uo_lookup[sid] = {}
    uo_lookup[sid][int(row.hour)] = row.urine_output

print(f"\n  Lookup built for {len(uo_lookup):,} stays")

# Free memory
del uo, uo_hourly
import gc; gc.collect()
print("  Memory freed ✓")
print("Urine output processed ✓")

Processing urine output...
  Shape after processing : (1071247, 3)
  Stays covered          : 89,170
  Missing urine_output   : 0
  UO stats (mL/hr):
    Mean  : 135.9
    Median: 80.0
    Max   : 3000.0
  Sample:
    stay_id  hour  urine_output
0  30000153     0         280.0
1  30000153     1          45.0
2  30000153     2          50.0
3  30000153     3          50.0
4  30000153     4          45.0

  Lookup built for 89,170 stays
  Memory freed ✓
Urine output processed ✓


## Processing Vasorepressors

In [9]:
# ── Cell 6: Process vasopressors into hourly lookup ───────────
print("Loading vasopressors...")

vaso = pd.read_csv(OUTPUT_DIR / 'vasopressors_filtered.csv')
print(f"  Raw shape  : {vaso.shape}")
print(f"  Columns    : {vaso.columns.tolist()}")
print(f"  Sample:")
print(vaso.head(3).to_string())

Loading vasopressors...
  Raw shape  : (765242, 6)
  Columns    : ['stay_id', 'starttime', 'itemid', 'amount', 'rate', 'rateuom']
  Sample:
    stay_id            starttime  itemid    amount      rate     rateuom
0  37081114  2150-11-02 20:00:00  221749  5.475664  0.600106  mcg/kg/min
1  37081114  2150-11-02 22:45:00  221749  1.410112  0.499987  mcg/kg/min
2  37081114  2150-11-02 23:36:00  221749  1.305181  0.400031  mcg/kg/min


In [11]:
# ── Cell 6: Process vasopressors into hourly lookup ───────────
print("Processing vasopressors...")

vaso['starttime'] = pd.to_datetime(vaso['starttime'])

# Merge intime
vaso = vaso.merge(
    cohort[['stay_id', 'intime']], on='stay_id', how='left'
)
vaso['intime'] = pd.to_datetime(vaso['intime'])

# Compute hour since admission
vaso['hour'] = (
    (vaso['starttime'] - vaso['intime'])
    .dt.total_seconds() / 3600
).astype(int)

# Keep first 24h only
vaso = vaso[
    (vaso['hour'] >= 0) &
    (vaso['hour'] < 24)
].copy()

# Binary flag — if any vasopressor was given in that hour = 1
vaso_hourly = (
    vaso
    .groupby(['stay_id', 'hour'])
    .size()
    .reset_index(name='vaso_count')
)
vaso_hourly['vasopressor_flag'] = 1

print(f"  Shape after processing : {vaso_hourly.shape}")
print(f"  Stays on vasopressors  : {vaso_hourly['stay_id'].nunique():,}")
print(f"  Sample:")
print(vaso_hourly.head(5).to_string())

# Build lookup: {stay_id: set of hours with vasopressors}
# Hours NOT in the set = 0 (no vasopressor)
# Hours IN the set     = 1 (vasopressor given)
vaso_lookup = {}
for row in vaso_hourly.itertuples():
    sid = row.stay_id
    if sid not in vaso_lookup:
        vaso_lookup[sid] = set()
    vaso_lookup[sid].add(int(row.hour))

print(f"\n  Lookup built for {len(vaso_lookup):,} stays")

# Free memory
del vaso, vaso_hourly
import gc; gc.collect()
print("  Memory freed ✓")
print("Vasopressors processed ✓")

Processing vasopressors...
  Shape after processing : (161771, 4)
  Stays on vasopressors  : 24,032
  Sample:
    stay_id  hour  vaso_count  vasopressor_flag
0  30000484    13           2                 1
1  30000484    16           1                 1
2  30000646     7           2                 1
3  30000646    14           1                 1
4  30000646    15           1                 1

  Lookup built for 24,032 stays
  Memory freed ✓
Vasopressors processed ✓


## Main Tensor Build 

In [ ]:
# ── Cell 7: Build rich hourly tensor ──────────────────────────
print("Building rich hourly tensor...")
print(f"  Stays      : {len(stay_ids_order):,}")
print(f"  Hours      : 24")
print(f"  Features   : {len(RICH_FEATURE_COLS)}")
print(f"  Shape      : ({len(stay_ids_order)}, 24, {len(RICH_FEATURE_COLS)})")

# Estimate memory
n_stays   = len(stay_ids_order)
n_hours   = 24
n_features= len(RICH_FEATURE_COLS)
mem_mb    = n_stays * n_hours * n_features * 4 / 1e6
print(f"  Memory est : {mem_mb:.1f} MB")

# Initialize tensor with NaN
X_rich = np.full(
    (n_stays, n_hours, n_features),
    fill_value=np.nan,
    dtype=np.float32
)

# Feature indices
IDX_ABP_DIA    = 0
IDX_ABP_MEAN   = 1
IDX_ABP_SYS    = 2
IDX_HR         = 3
IDX_RR         = 4
IDX_SPO2       = 5
IDX_TEMP       = 6
IDX_LACTATE    = 7
IDX_WBC        = 8
IDX_SOFA_PLT   = 9
IDX_SOFA_BILI  = 10
IDX_SOFA_CREAT = 11
IDX_UO         = 12
IDX_VASO       = 13
IDX_SHOCK      = 14
IDX_HR_DELTA   = 15
IDX_TEMP_DEV   = 16
IDX_CRP           = 17
IDX_PLT_RAW       = 18
IDX_INR           = 19
IDX_LACTATE_FRESH = 20
IDX_WBC_FRESH     = 21
IDX_CRP_FRESH     = 22
IDX_PLT_FRESH     = 23
IDX_INR_FRESH     = 24

print("\nFilling tensor...")
skipped = 0

for stay_id in tqdm(stay_ids_order):
    idx = stay_to_idx[stay_id]

    # ── 1. Copy vitals from X_vitals (0-6) ────────────────────
    X_rich[idx, :, :7] = X_vitals[stay_to_idx[stay_id], :, :]

    # ── 2. Lactate and WBC (7-8) ──────────────────────────────
    # ── 2. Sparse labs: values + freshness flags (7-8, 17-24) ─
if stay_id in labs_lookup:
    for hour, vals in labs_lookup[stay_id].items():
        if 0 <= hour < 24:
            # Values — NaN if not measured this hour
            X_rich[idx, hour, IDX_LACTATE]  = vals['lactate']
            X_rich[idx, hour, IDX_WBC]      = vals['wbc']
            X_rich[idx, hour, IDX_CRP]      = vals['crp']
            X_rich[idx, hour, IDX_PLT_RAW]  = vals['platelets_raw']
            X_rich[idx, hour, IDX_INR]      = vals['inr']

            # Freshness flags — 1 only in hours where lab was actually drawn
            X_rich[idx, hour, IDX_LACTATE_FRESH] = 0 if np.isnan(vals['lactate'])      else 1
            X_rich[idx, hour, IDX_WBC_FRESH]     = 0 if np.isnan(vals['wbc'])          else 1
            X_rich[idx, hour, IDX_CRP_FRESH]     = 0 if np.isnan(vals['crp'])          else 1
            X_rich[idx, hour, IDX_PLT_FRESH]     = 0 if np.isnan(vals['platelets_raw'])else 1
            X_rich[idx, hour, IDX_INR_FRESH]     = 0 if np.isnan(vals['inr'])          else 1

# Fill freshness flag NaNs with 0 — no measurement = 0, not unknown
for fresh_idx in [IDX_LACTATE_FRESH, IDX_WBC_FRESH, IDX_CRP_FRESH,
                  IDX_PLT_FRESH, IDX_INR_FRESH]:
    col = X_rich[idx, :, fresh_idx]
    X_rich[idx, :, fresh_idx] = np.where(np.isnan(col), 0.0, col)

    # ── 3. SOFA scores (9-11) ─────────────────────────────────
    if stay_id in sofa_lookup:
        for hour, vals in sofa_lookup[stay_id].items():
            if 0 <= hour < 24:
                X_rich[idx, hour, IDX_SOFA_PLT]   = vals['sofa_platelets']
                X_rich[idx, hour, IDX_SOFA_BILI]  = vals['sofa_bilirubin']
                X_rich[idx, hour, IDX_SOFA_CREAT] = vals['sofa_creatinine']

    # ── 4. Urine output (12) ──────────────────────────────────
    if stay_id in uo_lookup:
        for hour, val in uo_lookup[stay_id].items():
            if 0 <= hour < 24:
                X_rich[idx, hour, IDX_UO] = val

    # ── 5. Vasopressor flag (13) ──────────────────────────────
    if stay_id in vaso_lookup:
        for hour in vaso_lookup[stay_id]:
            if 0 <= hour < 24:
                X_rich[idx, hour, IDX_VASO] = 1.0
    # Hours with no vasopressor stay NaN → imputed to 0 in file 3
    # But for vaso flag, NaN means 0 so fill directly
    vaso_col = X_rich[idx, :, IDX_VASO]
    X_rich[idx, :, IDX_VASO] = np.where(np.isnan(vaso_col), 0.0, vaso_col)

    # ── 6. Derived features (14-16) ───────────────────────────
    hr   = X_rich[idx, :, IDX_HR].copy()
    sbp  = X_rich[idx, :, IDX_ABP_SYS].copy()
    temp = X_rich[idx, :, IDX_TEMP].copy()

    # Shock index = HR / SBP (NaN if either is NaN)
    with np.errstate(divide='ignore', invalid='ignore'):
        shock = np.where(sbp > 0, hr / sbp, np.nan)
    X_rich[idx, :, IDX_SHOCK] = shock

    # HR delta = HR[t] - HR[t-1] (NaN for hour 0)
    hr_delta       = np.full(24, np.nan, dtype=np.float32)
    hr_delta[1:]   = np.diff(hr)
    X_rich[idx, :, IDX_HR_DELTA] = hr_delta

    # Temp deviation from normal = |temp - 37.0|
    X_rich[idx, :, IDX_TEMP_DEV] = np.abs(temp - 37.0)

print("\nTensor filled ✓")

# ── Verification ──────────────────────────────────────────────
print(f"\nShape              : {X_rich.shape}")
print(f"Total elements     : {X_rich.size:,}")
print(f"NaN count per feature:")
for i, name in enumerate(RICH_FEATURE_COLS):
    nan_pct = np.isnan(X_rich[:, :, i]).mean() * 100
    print(f"  {name:<20} : {nan_pct:.1f}% NaN")

# Memory usage
print(f"\nMemory used        : {X_rich.nbytes / 1e6:.1f} MB")

Building rich hourly tensor...
  Stays      : 74,550
  Hours      : 24
  Features   : 17
  Shape      : (74550, 24, 17)
  Memory est : 121.7 MB

Filling tensor...


100%|██████████| 74550/74550 [00:00<00:00, 75160.09it/s]



Tensor filled ✓

Shape              : (74550, 24, 17)
Total elements     : 30,416,400
NaN count per feature:
  abp_dia              : 64.5% NaN
  abp_mean             : 64.3% NaN
  abp_sys              : 64.5% NaN
  heart_rate           : 0.0% NaN
  resp_rate            : 0.2% NaN
  spo2                 : 0.0% NaN
  temp_c               : 91.8% NaN
  lactate              : 94.4% NaN
  wbc                  : 92.0% NaN
  sofa_platelets       : 91.9% NaN
  sofa_bilirubin       : 97.8% NaN
  sofa_creatinine      : 92.0% NaN
  urine_output         : 54.9% NaN
  vasopressor_flag     : 0.0% NaN
  shock_index          : 64.5% NaN
  hr_delta             : 4.2% NaN
  temp_deviation       : 91.8% NaN

Memory used        : 121.7 MB


In [13]:
# ── Cell 8: Save tensor and metadata ──────────────────────────
print("Saving rich tensor...")

# Save tensor
np.save(str(OUTPUT_DIR / 'X_rich.npy'), X_rich)
print(f"  X_rich.npy saved     : {X_rich.shape}")

# Save feature names for reference in file 3
with open(OUTPUT_DIR / 'rich_feature_names.txt', 'w') as f:
    for name in RICH_FEATURE_COLS:
        f.write(name + '\n')
print(f"  rich_feature_names.txt saved : {len(RICH_FEATURE_COLS)} features")

# Save stay_ids_order so file 3 uses the same order
pd.Series(stay_ids_order).to_csv(
    OUTPUT_DIR / 'stay_ids_order.csv', index=False, header=True
)
print(f"  stay_ids_order.csv saved : {len(stay_ids_order):,} stays")

# Verify saved file
X_check = np.load(str(OUTPUT_DIR / 'X_rich.npy'))
assert X_check.shape == X_rich.shape, "Shape mismatch after save!"
print(f"\nVerification passed ✓")
print(f"  Shape  : {X_check.shape}")
print(f"  dtype  : {X_check.dtype}")
print(f"  Size   : {X_check.nbytes / 1e6:.1f} MB on disk")

del X_check
import gc; gc.collect()

print("\n" + "="*55)
print("02b_rich_tensor.ipynb complete ✓")
print("="*55)
print(f"\nOutputs saved to:")
print(f"  {OUTPUT_DIR / 'X_rich.npy'}")
print(f"  {OUTPUT_DIR / 'rich_feature_names.txt'}")
print(f"  {OUTPUT_DIR / 'stay_ids_order.csv'}")
print(f"\nNext: Load X_rich.npy in 03_model_training.ipynb")
print(f"      Replace X_vitals with X_rich in LSTM pipeline")

Saving rich tensor...
  X_rich.npy saved     : (74550, 24, 17)
  rich_feature_names.txt saved : 17 features
  stay_ids_order.csv saved : 74,550 stays

Verification passed ✓
  Shape  : (74550, 24, 17)
  dtype  : float32
  Size   : 121.7 MB on disk

02b_rich_tensor.ipynb complete ✓

Outputs saved to:
  /mnt/c/Users/20220460/Desktop/mimic_iv_processed/X_rich.npy
  /mnt/c/Users/20220460/Desktop/mimic_iv_processed/rich_feature_names.txt
  /mnt/c/Users/20220460/Desktop/mimic_iv_processed/stay_ids_order.csv

Next: Load X_rich.npy in 03_model_training.ipynb
      Replace X_vitals with X_rich in LSTM pipeline


Bad pipe message: %s [b'"Chromium";v="146", "Not-A.Brand";v="24", "Microsoft Edge']
Bad pipe message: %s [b'v="146"\r\nsec-ch-ua-mobile: ?0\r\nse', b'ch-ua-platform: "Windows"\r\nUpgrade-Insecure-Requests: 1\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0;', b'in64; x64) AppleWebKit/537.36 (', b'TML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0\r\nAccept: tex']
Bad pipe message: %s [b'html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exch', b'ge;v=b3;q=0.7\r\nSec-Fetch-Site: none\r\nSec-Fetch-Mode: navigate\r\nSec-Fetch-User: ?1\r\nSec-Fetch-Des']
Bad pipe message: %s [b'ol: max-age=0\r\nsec-ch-ua: "Chromium";v="146", "Not-A.Brand";v="24", "Microsoft Edge";v="146"\r\nsec-ch-ua-mobile: ?0\r']
Bad pipe message: %s [b'ec-ch-ua-', b'atform: "Windows"\r\nUpgrade-Insecure-Requests: 1\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb']
Bad pipe message: %s [b't/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safa